In [0]:
%pip install -r /Workspace/Users/omaku.andy609@gmail.com/ai-research-copilot/embedding_service/requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
import tempfile
from pathlib import Path
from databricks.sdk import WorkspaceClient

# Your Alfred Git folder.
repo = "/Workspace/Users/omaku.andy609@gmail.com/ai-research-copilot"

# Your existing saved model.
source = "/Volumes/workspace/default/models/modernbert-embed-base"

sys.path.insert(0, str(Path(repo) / "embedding_service"))

from staging import stage_model

local = str(Path(tempfile.mkdtemp()) / "model")

fingerprint = stage_model(
    source,
    local,
    WorkspaceClient(),
)

print("Local model directory:", local)
print("Model fingerprint:", fingerprint)

In [0]:
import runpy

def run_probe(arguments):
    previous_arguments = sys.argv[:]

    try:
        sys.argv = ["probe.py", *arguments]
        runpy.run_path(
            str(Path(repo) / "embedding_service" / "probe.py"),
            run_name="__main__",
        )
    finally:
        sys.argv = previous_arguments

In [0]:
run_probe([
    "--model-path", local,
    "--output", "/tmp/embedding-local.json",
])

In [0]:
import requests
from databricks.sdk import WorkspaceClient

app_url = "https://alfred-embeddings-probe-7474651285449311.aws.databricksapps.com"

client = WorkspaceClient()

response = requests.get(
    app_url.rstrip("/") + "/api/ready",
    headers=client.config.authenticate(),
    timeout=30,
    allow_redirects=False,
)

print("HTTP status:", response.status_code)
print(response.text)

In [0]:
import importlib
import sys

import requests
from databricks.sdk import WorkspaceClient

# The project modules live in the embedding_service folder next to this
# notebook; make them importable regardless of kernel state (cell 3's
# sys.path insert does not survive a kernel restart).
sys.path.insert(0, "/Workspace/Users/omaku.andy609@gmail.com/ai-research-copilot/embedding_service")

from notebook_auth import NotebookAppAuth
import probe

importlib.reload(probe)

app_url = (
    "https://alfred-embeddings-probe-7474651285449311"
    ".aws.databricksapps.com"
)

client = WorkspaceClient()

def notebook_token():
    return (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )

app_auth = NotebookAppAuth(
    client,
    "alfred-embeddings-probe",
    notebook_token,
)

response = requests.get(
    app_url + "/api/ready",
    headers=app_auth.authenticate(),
    timeout=30,
    allow_redirects=False,
)

print("HTTP status:", response.status_code)
print(response.text)